## Using MobileNetV2 for Face Mask Detection

> Importing libraries
---

In [ ]:
import pandas as pd
import os
import cv2
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

> Dataset Inspection
---

In [32]:
df = pd.read_csv("dataset/train/_classes.csv")

print(df.head())
print(df.columns.tolist())
print(df.shape)
print(df.sum(numeric_only=True))

                                            filename   with_mask  \
0  26-with-mask_jpg.rf.fb2a4fd14461ba4e7d981f51d9...           1   
1     66_jpg.rf.fb48333563aba0cd06afba343f625d98.jpg           0   
2  153-with-mask_jpg.rf.fb584ce29ce6d9a3123a04d80...           1   
3  29-with-mask_jpg.rf.fb9ecac8d3dc335a43b79a9438...           1   
4  augmented_image_315_jpg.rf.fba9c25c6675c5a039f...           1   

    without_mask  
0              0  
1              1  
2              0  
3              0  
4              0  
['filename', ' with_mask', ' without_mask']
(1279, 3)
with_mask       644
without_mask    635
dtype: int64


In [34]:
df.columns = df.columns.str.strip()

# Image directory
IMAGE_DIR = "dataset/train/"

# Check image availability
df["exists"] = df["filename"].apply(
    lambda x: os.path.exists(os.path.join(IMAGE_DIR, x))
)

print("Total images:", len(df))
print("Existing images:", df["exists"].sum())
print("Missing images:", (~df["exists"]).sum())

print("\nClass distribution:")
print(df[["with_mask", "without_mask"]].sum())

Total images: 1279
Existing images: 1279
Missing images: 0

Class distribution:
with_mask       644
without_mask    635
dtype: int64


> Train Test Split
---

In [35]:
IMAGE_DIR = "dataset/train"
IMG_SIZE = 160

df = pd.read_csv(os.path.join(IMAGE_DIR, "_classes.csv"))
df.columns = df.columns.str.strip()

X = []
y = []

for _, row in df.iterrows():
    img_path = os.path.join(IMAGE_DIR, row["filename"])

    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

    X.append(img)

    label = 1 if row["with_mask"] == 1 else 0
    y.append(label)

X = np.array(X, dtype=np.float32) / 255.0
y = np.array(y, dtype=np.int32)

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)

Training: (1023, 160, 160, 3) (1023,)
Validation: (256, 160, 160, 3) (256,)
